# Annotation listing and navigation

This notebook imports pUC19 from a GenBank file, loads its feature annotations,
and navigates to the MCS (multiple cloning site) using `widget.go_to()` and
`widget.show()`.

In [1]:
import os
import tempfile
import gen

## Import pUC19

Create a fresh in-memory repository, import a GenBank file and start a new plot figure (or widget in Jupyter terms).

In [2]:
# Adjust this path if running from outside the project root.
FIXTURE = os.path.abspath("../../fixtures/puc19.gb")

tmp = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmp, ".gen"))
repo.import_genbank(FIXTURE)

bg = repo.get_block_groups()[0]
print("Block group:", bg.name)

fig = bg.plot(rows=24)

Block group: sequence-222046-


## Annotations
Annotation groups associated with the sequence or its ancestors are automatically loaded as an annotation track titled "Stored annotations"

Additional GFF3 or BED annotation files can be loaded at any time:

```python
fig.add_annotation_track(file="path/to/features.gff3")
```

To start with a blank canvas before loading your own files, clear the auto-loaded tracks:

```python
fig.clear_all_annotations()
```

## List all annotations

`fig.list_annotations()` returns `Annotation` objects spanning all
sources (track panels and inline highlights).  Each annotation exposes `.name`,
`.locus`, and can be passed directly to `fig.go_to()` or `fig.show()`.

In [3]:
anns = fig.list_annotations()
print(f"{len(anns)} annotations loaded")
for a in anns:
    print(a)

21 annotations loaded
source
pBR322ori-F
L4440
CAP binding site
lac promoter
lac operator
M13/pUC Reverse
M13 rev
M13 Reverse
lacZ-alpha
MCS
M13 Forward
M13 fwd
M13/pUC Forward
pRS-marker
pGEX 3'
pBRforEco
AmpR promoter
AmpR
Amp-R
ori


## Navigate to the MCS

Filter for the MCS feature and navigate to it two ways:

* `widget.go_to(ann)` — left-pins the annotation start at column 12, no highlight
* `widget.show(ann)` — left-pins the annotation start and adds a highlight

In [4]:
# Left-pin the MCS start at column 12 from the left edge.
mcs = next(a for a in anns if a.name == "MCS")
fig.go_to(mcs)
fig


In [5]:
# Or go somewhere and immediately add a highlight.
pbla = next(a for a in anns if a.name == "AmpR promoter")

fig.show(pbla)
fig

## Search, filter, and navigate

Search for a sequence, wrap matches as `Annotation` objects, then navigate to one.

In [6]:
fig = bg.plot(rows=24)

# Search for Dcm methylation sites and add matches to the widget.
results = bg.search("CCWGG")

# The search query is a palindrome, so every match will also be a match on the opposite strand,
# filter by strand to prevent double labels:
matches = [locus for locus in results if locus.strand == "+"]
print(f"{len(matches)} match(es) found")

for i, locus in enumerate(matches):
    fig.add_annotation(gen.Annotation(locus, f"Dcm ({i + 1} of {len(matches)})"), )

fig.go_to(matches[0])
fig

5 match(es) found


In [7]:
# List all annotations now in the widget, then navigate to the first hit.
anns2 = fig.list_annotations()
print(f"{len(anns2)} total annotations")

hits = [a for a in anns2 if a.name.startswith("Dcm")]
print(f"{len(hits)} Dcm hit(s)")


fig.go_to(hits[0])
fig

26 total annotations
5 Dcm hit(s)


## Offset positions: `annotation - n` and `annotation + n`

`annotation - n` and `annotation + n` create an `AnnotationOffset` — a request to find
all graph positions *n* bases upstream or downstream of an annotation's start.

Passing the result to `widget.show()` highlights every reachable position in one colour
without moving the camera.  On a linear sequence there is one position; on a branched
graph (e.g. a variant site nearby) all branches light up.

In [8]:
# Fresh plot centred on the MCS.
fig = bg.plot(rows=24)

mcs = next(a for a in fig.list_annotations() if a.name == "MCS")

# Highlight the position 5 bp upstream of the MCS start.
# annotation - n  →  AnnotationOffset(annotation, -5)
fig.show(mcs - 5)
fig

## Bifurcating cursor:


In [9]:
# Downstream: highlight 200 bp after the AmpR promoter start.
ampR_p = next(a for a in fig.list_annotations() if a.name == "AmpR promoter")
fig.show(ampR_p + 200)
fig

In [10]:
repo.import_gfa(os.path.abspath('../../fixtures/anderson_promoters.gfa'), sample='library')


"'/Users/bvh/git/gen/fixtures/anderson_promoters.gfa' imported."

In [27]:
bg = repo.get_block_groups()[-1]
fig = bg.plot()
match = bg.search('TAGCTACTAGTGAAA')
ann = gen.Annotation(match[0], 'test')
fig.add_annotation(ann)

fig.show(ann-5)
fig